# Lab 05 — Mini-project: the Enterprise Operations Assistant (solution)

> Reference solution: every TODO is filled in and every check passes.

**Level:** Intermediate → Advanced · **Duration:** ~75 minutes · **Covers:** H9 Mini-project: incident triage → knowledge lookup → structured escalation output (60 min) · H10 architecture review: context-window risks and bottlenecks (15 min)

Lab guide: [`../lab-05-enterprise-operations-assistant.md`](../lab-05-enterprise-operations-assistant.md) — read each part there first; this notebook holds the code.

Design and verify: you choose the architecture, implement it, enforce the policy in code, measure it on both providers, and justify the choice in a written review. The finished reference is packaged in `ops_assistant/` for Module 5.

In [ ]:
# (notebook) Setup: run this cell first. From this lab on, the provider comes from LLM_PROVIDER via labs/setup/llm.py.
import importlib.util
import json
import os
import re
import sqlite3
import sys
import time
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))  # walks up from this folder to the repo-root .env
pd.set_option("display.max_colwidth", 110)

LAB_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents) if p.name == "module-02" and (p / "data").is_dir())
DATA_DIR, OUTPUT_DIR = LAB_DIR / "data", LAB_DIR / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
sys.path.append(str(LAB_DIR.parent / "setup"))
from llm import ProviderConfigError, describe, get_chat_model, get_provider  # noqa: E402

# Low reasoning effort on both providers; never pass temperature to Claude (current models reject it).
MODEL_KWARGS = {"openai": {"reasoning": {"effort": "low"}}, "anthropic": {"output_config": {"effort": "low"}, "max_tokens": 8_000}}


def make_model(provider: str | None = None, *, small: bool = False):
    """LangChain chat model for `provider` (default: LLM_PROVIDER), or None if it is not configured."""
    name = get_provider(provider)
    try:
        return get_chat_model(name, small=small, **MODEL_KWARGS[name])
    except ProviderConfigError as exc:
        print(f"⚠️  {exc}")
        return None


def check(name: str, fn) -> None:
    """Run a check function. A returned string means 'skipped'; a TODO is reported, not raised."""
    try:
        skipped = fn()
    except NotImplementedError as exc:
        print(f"⏳ {name}: TODO not implemented yet ({exc})")
        return
    print(f"⚠️  {name}: skipped ({skipped})" if skipped else f"✅ {name}")


PROVIDER = get_provider()
OTHER = "anthropic" if PROVIDER == "openai" else "openai"
llm, other_llm = make_model(PROVIDER), make_model(OTHER)

# Build the synthetic operations database once (data/build_ops_db.py; the .db file is git-ignored).
spec = importlib.util.spec_from_file_location("build_ops_db", DATA_DIR / "build_ops_db.py")
build_ops_db = importlib.util.module_from_spec(spec)
spec.loader.exec_module(build_ops_db)
DB_PATH = build_ops_db.build_ops_db(DATA_DIR / "ops.db")

TRACING = os.getenv("LANGSMITH_TRACING", "").lower() == "true" and bool(os.getenv("LANGSMITH_API_KEY"))
print(describe())
print(f"Comparison provider: {OTHER} {'✅' if other_llm else '❌ not configured'}")
print(f"Database: {DB_PATH.relative_to(LAB_DIR)} · LangSmith tracing: {'on' if TRACING else 'off (local JSONL trace instead)'}")

# Lab 05 builds on Lab 04's read-only tools and Lab 03's spec. The tools and schemas are packaged in
# labs/module-02/ops_assistant/ so that Module 5 can import the finished assistant.
from concurrent.futures import ThreadPoolExecutor

sys.path.insert(0, str(LAB_DIR))
from ops_assistant import tools as ops_tools  # noqa: E402
from ops_assistant.assistant import render_system_prompt  # noqa: E402
from ops_assistant.schemas import MATRIX, ROUTING, EscalationRecord, Triage  # noqa: E402

MODELS_BY_PROVIDER = {name: model for name, model in ((PROVIDER, llm), (OTHER, other_llm)) if model is not None}
PRICES = json.loads((DATA_DIR / "model_prices.json").read_text(encoding="utf-8"))["models"]


def run_parallel(fn, jobs: list[tuple], max_workers: int = 6) -> list:
    """Call fn(*job) for every job on a small thread pool; results come back in job order."""
    with ThreadPoolExecutor(max_workers=max_workers) as pool:
        return list(pool.map(lambda job: fn(*job), jobs))


print(f"Providers for the evaluation: {list(MODELS_BY_PROVIDER) or 'none configured'}")

## Part 1 — The brief: your spec, the contract and the evaluation set (5 min)

The assistant takes one incident ticket and returns an `EscalationRecord`: triage, knowledge
lookup and a structured escalation decision. Its system prompt is rendered from your Lab 03 spec
(or the reference spec if yours is unfinished). Eight labelled cases decide whether it works.

In [ ]:
# (notebook) The spec becomes the system prompt; the eval set becomes the definition of "works".
POLICY = (DATA_DIR / "triage_policy.md").read_text(encoding="utf-8")
TEAM_SPEC = OUTPUT_DIR / "enterprise-ops-assistant.spec.md"
REFERENCE_SPEC = LAB_DIR / "solutions" / "enterprise-ops-assistant.spec.md"
team_ready = TEAM_SPEC.exists() and "TODO" not in TEAM_SPEC.read_text(encoding="utf-8")
SPEC_PATH = TEAM_SPEC if team_ready else REFERENCE_SPEC
SYSTEM_PROMPT = render_system_prompt(SPEC_PATH.read_text(encoding="utf-8"), POLICY)
print(f"Spec: {SPEC_PATH.relative_to(LAB_DIR)}{'' if team_ready else ' (your outputs/ spec is missing or has TODOs)'}")
print(f"System prompt: {len(SYSTEM_PROMPT):,} characters\n")

CASES = json.loads((DATA_DIR / "eval_incidents.json").read_text(encoding="utf-8"))["cases"]
display(pd.DataFrame([{"case": c["case_id"], "ticket": c["text"][:70] + "…", **c["expected"]} for c in CASES]))
print("EscalationRecord fields:", ", ".join(EscalationRecord.model_fields))

## Part 2 — Build it: an agent or a workflow? (25 min)

Two architectures meet the same contract. **Agent:** one `create_agent` loop; the model decides
which read-only tools to call. **Workflow:** your code runs three fixed steps (triage call →
deterministic KB search → escalation call). Implement at least one; the harness below runs
whichever you list in `ARCHITECTURES`.

In [ ]:
# (notebook) The harness: runs one case through one architecture and records what it cost.
from dataclasses import dataclass, field

from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langchain.agents.structured_output import ProviderStrategy


@dataclass
class Result:
    provider: str
    architecture: str
    case_id: str
    record: EscalationRecord | None = None       # after enforce_policy (Part 3)
    raw: EscalationRecord | None = None          # what the model returned
    notes: list[str] = field(default_factory=list)
    tool_calls: list[str] = field(default_factory=list)
    model_calls: int = 0
    input_tokens: int = 0
    output_tokens: int = 0
    seconds: float = 0.0
    error: str | None = None


def ticket_message(text: str, ticket_id: str) -> str:
    return f'<ticket id="{ticket_id}">\n{text}\n</ticket>'


def count_usage(messages: list) -> tuple[int, int, int]:
    ai = [m for m in messages if getattr(m, "type", "") == "ai" and m.usage_metadata]
    return len(ai), sum(m.usage_metadata["input_tokens"] for m in ai), sum(m.usage_metadata["output_tokens"] for m in ai)


_agents: dict[str, object] = {}


def handle(provider: str, architecture: str, case: dict) -> Result:
    result = Result(provider, architecture, case["case_id"])
    model = MODELS_BY_PROVIDER[provider]
    started = time.perf_counter()
    try:
        if architecture == "agent":
            if provider not in _agents:
                _agents[provider] = build_agent(model)
            out = _agents[provider].invoke({"messages": [{"role": "user", "content": ticket_message(case["text"], case["ticket_id"])}]})
            result.raw, messages = out.get("structured_response"), out["messages"]
            result.tool_calls = [c["name"] for m in messages if m.type == "ai" for c in m.tool_calls]
        else:
            result.raw, messages = run_workflow(model, case["text"], case["ticket_id"])
            result.tool_calls = ["search_kb (by code)"]
        result.model_calls, result.input_tokens, result.output_tokens = count_usage(messages)
    except NotImplementedError:
        raise
    except Exception as exc:  # a failed run is a result to count, not a crash
        result.error = f"{exc.__class__.__name__}: {exc}"[:300]
    if result.raw is not None:
        policy = globals().get("enforce_policy")
        try:
            result.record, result.notes = policy(result.raw) if policy else (result.raw, ["policy not applied yet"])
        except NotImplementedError:
            result.record, result.notes = result.raw, ["policy not applied: enforce_policy is a TODO"]
    result.seconds = round(time.perf_counter() - started, 2)
    return result

In [ ]:
# (notebook) Your TODO: implement at least one architecture, then list it in ARCHITECTURES.
def build_agent(model):
    """Agent architecture: a create_agent loop the model drives.

    Contract: tools = ops_tools.langchain_tools(); system_prompt = SYSTEM_PROMPT;
    response_format = ProviderStrategy(EscalationRecord) (provider-native structured output);
    middleware = [ModelCallLimitMiddleware(run_limit=8, exit_behavior="error")] so a confused loop cannot run forever.
    """
    return create_agent(
        model,
        ops_tools.langchain_tools(),
        system_prompt=SYSTEM_PROMPT,
        response_format=ProviderStrategy(EscalationRecord),
        middleware=[ModelCallLimitMiddleware(run_limit=8, exit_behavior="error")],
    )


def run_workflow(model, text: str, ticket_id: str) -> tuple[EscalationRecord, list]:
    """Workflow architecture: your code decides the steps.

    1. Triage: model.with_structured_output(Triage, method="json_schema", include_raw=True) on
       [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": ticket_message(text, ticket_id)}].
    2. Lookup: articles = ops_tools.search_kb(text, 3). Code calls the tool, not the model.
    3. Escalation: model.with_structured_output(EscalationRecord, method="json_schema", include_raw=True) on the
       system prompt plus one user message holding the ticket, the triage JSON and the articles JSON, asking for
       the escalation record and to cite only KB IDs from the articles.
    Return (the parsed EscalationRecord, [raw AI message of step 1, raw AI message of step 3]).
    """
    system = {"role": "system", "content": SYSTEM_PROMPT}
    ticket = ticket_message(text, ticket_id)
    step1 = model.with_structured_output(Triage, method="json_schema", include_raw=True).invoke(
        [system, {"role": "user", "content": ticket}])
    articles = ops_tools.search_kb(text, 3)
    step3 = model.with_structured_output(EscalationRecord, method="json_schema", include_raw=True).invoke([
        system,
        {"role": "user", "content": f"{ticket}\n\n<triage>\n{step1['parsed'].model_dump_json()}\n</triage>\n\n"
                                    f"<search_kb_results>\n{json.dumps(articles)}\n</search_kb_results>\n\n"
                                    "Write the escalation record. Cite only KB IDs from the search results that apply."},
    ])
    return step3["parsed"], [step1["raw"], step3["raw"]]


ARCHITECTURES: list[str] = ["agent", "workflow"]

In [ ]:
# (notebook) Smoke test: one case through every architecture you listed, on LLM_PROVIDER.
smoke = {}
if llm and ARCHITECTURES:
    case = next(c for c in CASES if c["case_id"] == "EV-02")
    for architecture in ARCHITECTURES:
        r = smoke[architecture] = handle(PROVIDER, architecture, case)
        print(f"--- {architecture}: {r.model_calls} model calls · tools {r.tool_calls} · {r.input_tokens:,} in · {r.seconds}s · error={r.error}")
        print(r.raw.model_dump_json(indent=1) if r.raw else "no record")
elif not ARCHITECTURES:
    print("⏳ Implement build_agent() and/or run_workflow(), then set ARCHITECTURES.")


def check_part2():
    if not ARCHITECTURES:
        raise NotImplementedError("set ARCHITECTURES to the architecture(s) you implemented")
    assert set(ARCHITECTURES) <= {"agent", "workflow"}
    if not llm:
        return "no provider configured"
    for architecture, r in smoke.items():
        assert r.error is None, f"{architecture}: {r.error}"
        assert isinstance(r.raw, EscalationRecord), f"{architecture} returned no EscalationRecord"


check("Part 2 — the assistant runs", check_part2)

## Part 3 — Policy as code: what must always hold (15 min)

The model judges the category, impact, urgency and which article applies. Code enforces what the
spec says must **always** be true (E1, E2 and the output contract), and logs every correction.

In [ ]:
# (notebook) Given helpers: the allowed values, the known KB IDs and the "promised time" pattern.
RANK = {"P1": 1, "P2": 2, "P3": 3, "P4": 4}
KNOWN_KB = ops_tools.known_kb_ids()
PROMISED_TIME = re.compile(
    r"\b(will|'ll|shall)\b[^.!?]{0,60}?\b(within|in|by)\s+(\d+\s*(minutes?|mins?|hours?|hrs?|days?)|\d{1,2}[:.]\d{2})\b", re.IGNORECASE)


def allowed_priorities(category: str, impact: str, urgency: str) -> set[str]:
    """The matrix result, plus P2 for security (the floor for a confirmed incident)."""
    expected = MATRIX[(impact, urgency)]
    return {expected, "P2"} if category == "security" and RANK[expected] > 2 else {expected}


def allowed_groups(category: str, impact: str) -> set[str]:
    """The routing table, plus Platform Engineering for service-wide identity or email outages."""
    groups = {ROUTING[category]}
    return groups | {"Platform Engineering"} if category in ("access", "email_collab") and impact == "high" else groups

In [ ]:
# (notebook) Your TODO: enforce the policy on a record. Return a corrected COPY and one note per change.
def enforce_policy(record: EscalationRecord) -> tuple[EscalationRecord, list[str]]:
    """Rules, in this order (use record.model_copy(deep=True) and change the copy):

    1. priority not in allowed_priorities(...)  -> set it to the most urgent allowed value; note it.
    2. assignment_group not in allowed_groups(...) -> set ROUTING[category]; note it.
    3. kb_ids not in KNOWN_KB                    -> drop them; note it.
    4. priority P1/P2 or category security, but escalate is false -> escalate = True; note "E1"/"E2".
    5. escalate true and no escalation_target    -> "Security Operations" for security, else f"{group} L2".
       escalate false and a target               -> target = None.
    6. user_reply matches PROMISED_TIME          -> note starting with "warning:" (a human checks it; do not edit text).
    """
    r, notes = record.model_copy(deep=True), []
    priorities = allowed_priorities(r.category, r.impact, r.urgency)
    if r.priority not in priorities:
        fixed = min(priorities, key=RANK.get)
        notes.append(f"priority {r.priority} -> {fixed}")
        r.priority = fixed
    if r.assignment_group not in allowed_groups(r.category, r.impact):
        notes.append(f"group {r.assignment_group} -> {ROUTING[r.category]}")
        r.assignment_group = ROUTING[r.category]
    unknown = [k for k in r.kb_ids if k not in KNOWN_KB]
    if unknown:
        notes.append(f"dropped unknown KB {unknown}")
        r.kb_ids = [k for k in r.kb_ids if k in KNOWN_KB]
    if (RANK[r.priority] <= 2 or r.category == "security") and not r.escalate:
        notes.append("escalate -> true (" + ("E2 security" if r.category == "security" else f"E1 {r.priority}") + ")")
        r.escalate = True
    if r.escalate and not r.escalation_target:
        r.escalation_target = "Security Operations" if r.category == "security" else f"{r.assignment_group} L2"
        notes.append(f"target -> {r.escalation_target}")
    elif not r.escalate and r.escalation_target:
        notes.append("target cleared (escalate is false)")
        r.escalation_target = None
    if PROMISED_TIME.search(r.user_reply):
        notes.append("warning: the reply promises a time; route to human review")
    return r, notes

In [ ]:
# (notebook) Check: enforce_policy on hand-made records with known answers.
def record(**overrides) -> EscalationRecord:
    base = dict(incident_id="INC-104700", category="endpoint", impact="low", urgency="low", priority="P4",
                assignment_group="Service Desk", kb_ids=[], escalate=False, escalation_target=None,
                escalation_reason="none", user_reply="Thanks, an analyst will look at this.", next_actions=["Check the device"])
    return EscalationRecord(**{**base, **overrides})


def check_part3():
    r, notes = enforce_policy(record(category="security", urgency="high", priority="P3", kb_ids=["KB-0733", "KB-9999"]))
    assert (r.priority, r.assignment_group, r.escalate, r.escalation_target, r.kb_ids) == (
        "P3", "Security Operations", True, "Security Operations", ["KB-0733"]), r
    r, _ = enforce_policy(record(priority="P1", escalate=True, escalation_target="Service Desk L2"))
    assert r.priority == "P4", "low impact x low urgency is P4, whatever the model says"
    r, _ = enforce_policy(record(category="email_collab", impact="high", urgency="high", priority="P1",
                                 assignment_group="Platform Engineering", escalate=True))
    assert (r.assignment_group, r.escalation_target) == ("Platform Engineering", "Platform Engineering L2")
    r, notes = enforce_policy(record(user_reply="Your printer will be fixed within 2 hours."))
    assert any(n.startswith("warning") for n in notes)
    original = record(priority="P1")
    enforce_policy(original)
    assert original.priority == "P1", "enforce_policy must not modify its input"


check("Part 3 — policy as code", check_part3)

## Part 4 — Evaluate on both providers and choose (15 min)

Run every case through every architecture on both providers. Grade on structure and labels, never
on wording. Compare the raw model output with the policy-checked record: that gap is what Part 3 buys.

In [ ]:
# (notebook) 8 cases × your architectures × both providers.
def grade(rec: EscalationRecord | None, case: dict) -> dict:
    e = case["expected"]
    if rec is None:
        return {"valid": False, "priority": False, "group": False, "escalate": False, "kb": False, "all": False}
    g = {"valid": True, "priority": rec.priority == e["priority"], "group": rec.assignment_group == e["assignment_group"],
         "escalate": rec.escalate == e["escalate"], "kb": not e["kb_any_of"] or bool(set(rec.kb_ids) & set(e["kb_any_of"]))}
    return {**g, "all": all(g.values())}


def cost(r: Result) -> float | None:
    price = PRICES.get(os.getenv("OPENAI_MODEL" if r.provider == "openai" else "ANTHROPIC_MODEL", ""))
    return (r.input_tokens * price["input"] + r.output_tokens * price["output"]) / 1e6 if price else None


jobs = [(p, a, c) for p in MODELS_BY_PROVIDER for a in ARCHITECTURES for c in CASES]
started = time.perf_counter()
results = run_parallel(handle, jobs) if jobs else []
print(f"{len(results)} runs in {time.perf_counter() - started:.0f}s")

CASE_BY_ID = {c["case_id"]: c for c in CASES}
rows = []
for r in results:
    final, raw = grade(r.record, CASE_BY_ID[r.case_id]), grade(r.raw, CASE_BY_ID[r.case_id])
    rows.append({"provider": r.provider, "architecture": r.architecture, "case": r.case_id, **final, "raw all": raw["all"],
                 "corrections": sum(not n.startswith("warning") for n in r.notes), "notes": "; ".join(r.notes),
                 "seconds": r.seconds, "input tokens": r.input_tokens, "model calls": r.model_calls,
                 "cost ($)": cost(r), "error": r.error})
evaluation = pd.DataFrame(rows)
if not evaluation.empty:
    summary = evaluation.groupby(["provider", "architecture"]).agg(
        valid=("valid", "mean"), priority=("priority", "mean"), group=("group", "mean"), escalate=("escalate", "mean"),
        kb=("kb", "mean"), all_correct=("all", "mean"), raw_all_correct=("raw all", "mean"), corrections=("corrections", "sum"),
        avg_s=("seconds", "mean"), avg_input_tokens=("input tokens", "mean"), cost_per_ticket=("cost ($)", "mean"),
    ).round(3)
    display(summary)
    display(evaluation[~evaluation["all"]][["provider", "architecture", "case", "priority", "group", "escalate", "kb", "notes", "error"]])

In [ ]:
# (notebook) Check: every case ran everywhere and produced a valid record.
def check_part4():
    if not ARCHITECTURES:
        raise NotImplementedError("implement an architecture in Part 2")
    if evaluation.empty:
        return "no provider configured"
    assert len(evaluation) == len(MODELS_BY_PROVIDER) * len(ARCHITECTURES) * len(CASES)
    assert evaluation.valid.mean() >= 0.9, "more than 10% of runs returned no record: look at the error column"
    recall = evaluation[[CASE_BY_ID[c]["expected"]["escalate"] for c in evaluation.case]].escalate.mean()
    print(f"   escalation recall over all runs: {recall:.0%} (the spec asks for 100%)")


check("Part 4 — evaluation", check_part4)

📝 **Decision (3–5 sentences).** Which architecture and provider would you ship, and why? Cite the
numbers: accuracy, escalation recall, corrections, latency and cost per ticket. What did the
policy layer fix that the prompt did not?

> *Your answer here.*

## Part 5 — Architecture review: context-window risks and bottlenecks (15 min)

Measure where the tokens and the seconds go, push two inputs past what a real deployment will
see, and write the risks down with the evidence next to them.

In [ ]:
# (notebook) Where the first model call's input tokens go, and where one run's time goes.
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.messages.utils import count_tokens_approximately
from langchain_core.utils.function_calling import convert_to_openai_tool


class StepTimer(BaseCallbackHandler):
    """Collect (step, start, end) for every model call and tool call of one run."""

    def __init__(self):
        self.t0, self.open, self.steps = time.perf_counter(), {}, []

    def _start(self, run_id, label):
        self.open[run_id] = (label, time.perf_counter() - self.t0)

    def _end(self, run_id):
        label, start = self.open.pop(run_id, ("?", 0.0))
        self.steps.append({"step": label, "start (s)": round(start, 2), "duration (s)": round(time.perf_counter() - self.t0 - start, 2)})

    def on_chat_model_start(self, serialized, messages, *, run_id, **kwargs):
        self._start(run_id, "model call")

    def on_llm_end(self, response, *, run_id, **kwargs):
        self._end(run_id)

    def on_tool_start(self, serialized, input_str, *, run_id, **kwargs):
        self._start(run_id, f"tool {serialized.get('name')}")

    def on_tool_end(self, output, *, run_id, **kwargs):
        self._end(run_id)


budget, waterfall = pd.DataFrame(), pd.DataFrame()
if llm and "agent" in ARCHITECTURES:
    case = CASE_BY_ID["EV-04"]
    tool_schemas = [convert_to_openai_tool(t) for t in ops_tools.langchain_tools()]
    timer = StepTimer()
    out = build_agent(llm).invoke({"messages": [{"role": "user", "content": ticket_message(case["text"], case["ticket_id"])}]},
                                  {"callbacks": [timer]})
    first_call = next(m for m in out["messages"] if m.type == "ai" and m.usage_metadata)
    parts = {"system prompt (spec + policy)": count_tokens_approximately([{"role": "system", "content": SYSTEM_PROMPT}]),
             "tool schemas (4 tools)": round(len(json.dumps(tool_schemas)) / 4),
             "ticket": count_tokens_approximately([{"role": "user", "content": ticket_message(case["text"], case["ticket_id"])}])}
    parts["output schema + provider overhead"] = max(first_call.usage_metadata["input_tokens"] - sum(parts.values()), 0)
    budget = pd.DataFrame({"≈ tokens": parts}).assign(share=lambda d: (d["≈ tokens"] / d["≈ tokens"].sum()).round(2))
    waterfall = pd.DataFrame(timer.steps)
    print(f"First model call on {PROVIDER}: {first_call.usage_metadata['input_tokens']:,} input tokens (reported by the API)")
    display(budget)
    display(waterfall)

In [ ]:
# (notebook) Stress test 1: an unbounded tool result. Stress test 2: a ticket with a pasted log.
with sqlite3.connect(DB_PATH) as conn:
    all_rows = conn.execute("SELECT * FROM incidents").fetchall()
capped = ops_tools.query_ops_db("SELECT * FROM incidents")
approx = lambda obj: round(len(json.dumps(obj, default=str)) / 4)

LOG_LINES = [f"2026-10-27T07:{m // 60:02d}:{m % 60:02d}Z vpn-gateway-eu-01 auth user=u{m % 900:04d} result=ok latency_ms={40 + m % 17}"
             for m in range(2_400)]
for i in (1_210, 1_640, 2_101):
    LOG_LINES[i] = LOG_LINES[i].replace("result=ok", "result=FAIL error=809 reason=cached_credential")
BIG_TICKET = "VPN keeps failing for me after my password change. Full client log below.\n" + "\n".join(LOG_LINES)


def condense_attachment(text: str, head: int = 5, tail: int = 5, max_hits: int = 20) -> str:
    """Keep the first and last lines and every line that looks like an error; say what was dropped."""
    lines = text.splitlines()
    hits = [l for l in lines[head:-tail] if re.search(r"FAIL|ERROR|error=", l)][:max_hits]
    dropped = len(lines) - head - tail - len(hits)
    return "\n".join(lines[:head] + hits + [f"[… {dropped:,} routine lines removed by the gateway before the model call …]"] + lines[-tail:])


stress = [
    {"input": "query_ops_db: SELECT * FROM incidents, uncapped", "rows/lines": len(all_rows), "≈ tokens": approx(all_rows)},
    {"input": "query_ops_db: same query, 50-row cap", "rows/lines": len(capped["rows"]), "≈ tokens": approx(capped)},
    {"input": "ticket with pasted log, raw", "rows/lines": len(LOG_LINES), "≈ tokens": round(len(BIG_TICKET) / 4)},
    {"input": "ticket with pasted log, condensed", "rows/lines": len(condense_attachment(BIG_TICKET).splitlines()),
     "≈ tokens": round(len(condense_attachment(BIG_TICKET)) / 4)},
]
if llm and ARCHITECTURES:
    architecture = ARCHITECTURES[-1]
    for label, text in (("raw", BIG_TICKET), ("condensed", condense_attachment(BIG_TICKET))):
        r = handle(PROVIDER, architecture, {"case_id": f"STRESS-{label}", "ticket_id": "INC-104699", "text": text})
        stress.append({"input": f"{architecture} run on the {label} ticket ({PROVIDER})", "rows/lines": "—",
                       "≈ tokens": r.input_tokens, "seconds": r.seconds,
                       "outcome": r.record.priority + (" escalate" if r.record.escalate else "") if r.record else r.error})
stress_table = pd.DataFrame(stress)
stress_table

In [ ]:
# (notebook) Your review: write the decision and each risk's evidence and mitigation, then export the review.
from datetime import date

TEAM = "Reference solution"
REVIEW = {
    "decision": "Ship the agent architecture on the cheaper provider that clears the bar: both architectures reach the "
                "eval targets after the policy layer, the agent adapts to tickets that name incidents or services, and "
                "the policy layer makes the escalation decision deterministic where the spec requires it.",
    "r1_evidence": "SELECT * returns ~437 rows; uncapped that is tens of thousands of tokens, capped at 50 rows a fraction of it",
    "r1_mitigation": "Row cap in the tool, plus 'truncated: true' so the model narrows the query instead of guessing",
    "r1_owner": "Platform Engineering",
    "r2_evidence": "A 2,400-line pasted log is ~60k tokens raw and under 1k condensed; the raw run is slower and costs more",
    "r2_mitigation": "Condense attachments before the model call (head, tail, error lines) and store the full log by reference",
    "r2_owner": "Service Desk tooling",
    "r3_evidence": "The fixed prompt (spec + policy + tool schemas) is most of every first call's input",
    "r3_mitigation": "Prompt caching on the stable prefix; move rarely used rules into a Skill loaded on demand (Lab 03)",
    "r3_owner": "Assistant owner",
    "r4_evidence": "Model calls dominate the waterfall; tools return in milliseconds",
    "r4_mitigation": "Fewer model turns (workflow for simple tickets), low reasoning effort, stream the user reply",
    "r4_owner": "Assistant owner",
    "r5_evidence": "Knowledge lookup is keyword search over 10 articles; recall will drop as the KB grows",
    "r5_mitigation": "Replace search_kb with retrieval over embeddings (Module 3) and measure precision@k",
    "r5_owner": "Knowledge management",
    "next_steps": "Module 3: RAG over the full KB and policy documents behind the same search_kb contract. "
                  "Module 4: a supervisor that routes simple tickets to the workflow and complex ones to the agent, with a HITL gate on P1.",
}

In [ ]:
# (notebook) Export templates/architecture-review.md -> outputs/architecture-review.md.
def md_table(frame: pd.DataFrame) -> str:
    if frame is None or frame.empty:
        return "_not measured_"
    frame = frame.reset_index() if frame.index.name or not isinstance(frame.index, pd.RangeIndex) else frame
    cols = [str(c) for c in frame.columns]
    lines = ["| " + " | ".join(cols) + " |", "| " + " | ".join("---" for _ in cols) + " |"]
    lines += ["| " + " | ".join(str(v) for v in row) + " |" for row in frame.astype(object).fillna("—").to_numpy()]
    return "\n".join(lines)


def export_review() -> Path:
    values = {"team": TEAM, "date": date.today().isoformat(), "providers": ", ".join(MODELS_BY_PROVIDER) or "—",
              "architecture": ", ".join(ARCHITECTURES) or "—",
              "eval_table": md_table(summary) if not evaluation.empty else "_not run_",
              "budget_table": md_table(budget), "stress_table": md_table(stress_table), "waterfall_table": md_table(waterfall), **REVIEW}
    sheet = (LAB_DIR / "templates" / "architecture-review.md").read_text(encoding="utf-8")
    for key, value in values.items():
        sheet = sheet.replace("{{" + key + "}}", str(value))
    path = OUTPUT_DIR / "architecture-review.md"
    path.write_text(sheet, encoding="utf-8")
    return path


review_path = export_review()
print(f"Wrote {review_path.relative_to(LAB_DIR)}")


def check_part5():
    text = review_path.read_text(encoding="utf-8")
    assert "{{" not in text, "every placeholder must be filled"
    if "TODO" in text:
        raise NotImplementedError("fill in REVIEW (decision, R1-R5, next steps)")


check("Part 5 — architecture review", check_part5)

In [ ]:
# (notebook) The packaged reference (labs/module-02/ops_assistant/): Module 5 Lab 03 imports exactly this.
from ops_assistant import build_assistant

if llm:
    reference = build_assistant(PROVIDER, architecture="agent")
    for case in (CASE_BY_ID["EV-03"], CASE_BY_ID["EV-08"]):
        r = reference.handle(case["text"], case["ticket_id"])
        print(case["case_id"], "→", r.record.priority if r.record else r.error, "escalate" if r.record and r.record.escalate else "",
              r.record.assignment_group if r.record else "", "· notes:", r.policy_notes)

---

**Done.** Answer the checkpoint questions in the lab guide. That completes the Module 2 labs: see
the README's clean-up section, then **Module 3 — Enterprise RAG Pipelines & Fine-Tuning Strategy**.